# Self consistency of the model rater

Three cold passes over the same five papers, by the same model, in fresh contexts with no
memory between them. This measures how much of the model's disagreement with rater 1 is a
stable difference of interpretation and how much is run to run noise.

It is the one thing a model rater supplies that a human cannot. Nobody can be asked to read a
paper cold three times having forgotten the previous two.

---

## Rules carried in from D12 and D14

**Run 1 remains the primary run.** Nothing computed here changes the agreement figure of 0.686.
That was designated in advance and does not reopen, whatever runs 2 and 3 show.

**No majority voting into the primary comparison.** A modal verdict across the three runs is
computed below as a descriptive only. Substituting it for run 1 would inflate agreement against
a human who reads each paper once, and would be comparable to nothing.

**Computed on the raw mapped verdicts, before the quotation gate.** The gate exists to protect
the comparison against rater 1. Self consistency is a property of the model's own output, and
filtering it through a rule built for a different purpose would make the figure partly reflect
our transcription policy rather than the rater. Fixed before run 3 existed.

**The parser below is duplicated from `transcribe_model_rating.ipynb`.** The project keeps all
programmatic files as notebooks, so there is no shared module to import from. If the parser
changes in one place it must change in both.


In [ ]:
import re
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
RAW  = ROOT / 'Data' / 'Reliability' / 'Raw'
LONG = ROOT / 'Data' / 'Tracker' / 'exports' / 'scoring_long.csv'
EXP  = ROOT / 'Data' / 'Tracker' / 'exports'

RUNS = (1, 2, 3)
CATS = ['Yes', 'No', 'Unclear', 'N/A']

VERDICT = {'yes': 'Yes', 'no': 'No',
           'unclear': 'Unclear', 'uncertain': 'Unclear', 'cannot determine': 'Unclear',
           'not reported': 'Unclear', 'insufficient information': 'Unclear',
           'n/a': 'N/A', 'na': 'N/A', 'n.a.': 'N/A', 'not applicable': 'N/A'}

def norm(cell):
    s = re.sub(r'<[^>]*>', ' ', str(cell))
    s = re.sub(r'[*`_]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip().lower()

ROW_START = re.compile(r'^\s*\|\s*\**\s*\d')

def parse_transcript(path):
    lines = path.read_text(encoding='utf-8').split('\n')
    header, rows, cur, started = None, [], None, False
    for ln in lines:
        s = ln.strip()
        if not started:
            if s.startswith('|') and re.search(r'verdict', s, re.I):
                header, started = s, True
            continue
        if re.match(r'^(---|##)', s) and cur is not None and not s.startswith('|'):
            break
        if ROW_START.match(ln):
            if cur is not None:
                rows.append(cur)
            cur = s
        elif cur is not None:
            cur += ' ' + s
    if cur is not None:
        rows.append(cur)

    hdr = [norm(c) for c in header.strip().strip('|').split('|')]
    iv = next(i for i, h in enumerate(hdr) if 'verdict' in h)
    out = []
    for r in rows:
        cs = [c.strip() for c in r.strip().strip('|').split('|')]
        assert len(cs) == len(hdr), f'{path.name}: {len(cs)} cells vs header {len(hdr)}'
        m = re.search(r'\d+', cs[0])
        out.append({'item_no': int(m.group()), 'verdict': VERDICT.get(norm(cs[iv]))})
    return out

recs = []
for run in RUNS:
    files = sorted(RAW.rglob(f'run{run}_paper*.txt'))
    assert len(files) == 5, f'run {run}: found {len(files)} transcripts, expected 5'
    empty = [f.name for f in files if f.stat().st_size == 0]
    assert not empty, f'run {run}: empty transcript(s), nothing to parse: {empty}'
    for f in files:
        pid = int(re.search(r'paper(\d+)', f.name).group(1))
        for rec in parse_transcript(f):
            recs.append({'run': run, 'paper_id': pid, **rec})

raw = pd.DataFrame(recs)
assert raw.verdict.notna().all(), f'unmapped verdicts:\n{raw[raw.verdict.isna()]}'
assert len(raw) == 195, f'expected 195 cells across 3 runs, parsed {len(raw)}'

wide = raw.pivot(index=['paper_id', 'item_no'], columns='run', values='verdict')
wide.columns = [f'run{c}' for c in wide.columns]
wide = wide.reset_index()
assert len(wide) == 65 and wide.notna().all().all(), 'pivot is not a complete 65 by 3 grid'
print(f'parsed {len(raw)} verdicts, {len(wide)} cells rated {len(RUNS)} times each')


## How often does the rater agree with itself

Unanimity is the headline. Pairwise agreement and Fleiss' kappa are reported alongside it,
because on data skewed toward `Yes` no single coefficient is trustworthy, which is the same
reason three statistics were reported for the comparison against rater 1.


In [ ]:
RUNCOLS = [f'run{r}' for r in RUNS]

wide['n_distinct'] = wide[RUNCOLS].nunique(axis=1)
wide['unanimous'] = wide.n_distinct == 1

def fleiss_kappa(table):
    """table: rows are cells, columns are counts of raters per category."""
    n_raters = table.sum(axis=1)[0]
    P_i = (np.square(table).sum(axis=1) - n_raters) / (n_raters * (n_raters - 1))
    P_bar = P_i.mean()
    p_j = table.sum(axis=0) / (len(table) * n_raters)
    P_e = np.square(p_j).sum()
    return (P_bar - P_e) / (1 - P_e), P_bar, P_e

counts = np.array([[list(row).count(c) for c in CATS]
                   for row in wide[RUNCOLS].to_numpy()])
kf, pbar, pe = fleiss_kappa(counts)

pairs = []
for a, b in combinations(RUNCOLS, 2):
    pairs.append((f'{a} vs {b}', (wide[a] == wide[b]).mean()))

print('SELF CONSISTENCY')
print('=' * 62)
print(f'  cells rated three times          {len(wide)}')
print(f'  unanimous across all three runs  {int(wide.unanimous.sum())}'
      f'  ({wide.unanimous.mean():.1%})')
print(f'  two of three agreed              {int((wide.n_distinct == 2).sum())}')
print(f'  all three differed               {int((wide.n_distinct == 3).sum())}')
print()
for label, v in pairs:
    print(f'  {label:<28} {v:.1%}')
print(f"  {'mean pairwise agreement':<28} {np.mean([v for _, v in pairs]):.1%}")
print()
print(f"  Fleiss' kappa across runs        {kf:.3f}")
print(f'     observed agreement            {pbar:.3f}')
print(f'     expected by chance            {pe:.3f}')
print('=' * 62)


## Where the instability sits

A single overall figure hides whether the variance is spread evenly or concentrated in a few
items. If it concentrates, that names the part of the instrument a reader should distrust.


In [ ]:
ITEM = {1: 'External Validation', 2: 'Patient-Level RCT', 3: 'Prespecified Endpoint',
        4: 'Prespecified Design', 5: 'Registered w/ CT.gov', 6: 'Transparent Reporting',
        7: 'Honest Impact Eval', 8: 'No Conflict of Interest', 9: 'Focus on Patient Outcomes',
        10: 'Adequate Sample Size', 11: 'Intention-to-Treat', 12: 'Balanced Table 1',
        13: 'No Reverse Causation'}

by_item = (wide.groupby('item_no')
           .agg(n=('unanimous', 'size'), unanimous=('unanimous', 'sum'))
           .assign(rate=lambda d: d.unanimous / d.n)
           .sort_values(['rate', 'item_no']))
by_item.insert(0, 'item', by_item.index.map(ITEM))
print('UNANIMITY BY ITEM')
print(by_item.to_string())
print()
by_paper = (wide.groupby('paper_id')
            .agg(n=('unanimous', 'size'), unanimous=('unanimous', 'sum'))
            .assign(rate=lambda d: d.unanimous / d.n))
print('UNANIMITY BY PAPER')
print(by_paper.to_string())
print()
unstable = wide[~wide.unanimous].copy()
unstable['item'] = unstable.item_no.map(ITEM)
print(f'{len(unstable)} unstable cell(s):')
print(unstable[['paper_id', 'item_no', 'item'] + RUNCOLS].to_string(index=False))


## Does instability explain the disagreement with rater 1

This is the question the whole exercise exists to answer, and it was stated before run 3 was
run. If the cells that disagreed with rater 1 are also the cells the model cannot reproduce,
then part of that disagreement is noise rather than interpretation, and the reproducibility
claim weakens. If the disagreements are stable, they are real differences about what the
instrument means, and they are worth more.

Rater 1's verdicts are loaded here and only here. They are used to partition the cells, never to
alter one.


In [ ]:
r1 = pd.read_csv(LONG, keep_default_na=False)[['paper_id', 'item_no', 'verdict']]
r1 = r1.rename(columns={'verdict': 'rater1'})
m = wide.merge(r1, on=['paper_id', 'item_no'], how='left', validate='one_to_one')
assert m.rater1.ne('').all(), 'rater 1 verdict missing for some cell'

m['agreed_run1'] = m.run1 == m.rater1
tab = (m.groupby('agreed_run1')
       .agg(cells=('unanimous', 'size'), unanimous=('unanimous', 'sum'))
       .assign(rate=lambda d: d.unanimous / d.cells))
tab.index = ['run 1 DISAGREED with rater 1', 'run 1 agreed with rater 1']
print('STABILITY, SPLIT BY WHETHER RUN 1 AGREED WITH RATER 1')
print(tab.to_string())
print()

# modal verdict, DESCRIPTIVE ONLY, never substituted into the primary comparison
m['modal'] = m[RUNCOLS].mode(axis=1)[0]
print(f"run 1 agreed with rater 1 on          {int(m.agreed_run1.sum())} of {len(m)} cells")
print(f"the modal verdict would have agreed on {int((m.modal == m.rater1).sum())} of {len(m)}")
print('  (descriptive only. D12 forbids voting this into the reported figure.)')

m.drop(columns=['n_distinct']).to_csv(EXP / 'self_consistency.csv', index=False)
print(f"\nwrote {(EXP / 'self_consistency.csv').relative_to(ROOT)}")
